Kugelwolkenmodell-Chemie. Erstellt JSON Objekt.

Prokrustes Algorithmus. Sowas wie Kurven fitten. Auch inverse Kinematik. Andockproblem.

In [15]:
import numpy as np
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import uuid

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)



In [25]:

# ==============================================================================
# 1. DAS PERIODENSYSTEM DER GEOMETRIEN (Die "Bauanleitung")
# ==============================================================================
# Jedes Element definiert seinen Geometrie-Typ, die Gesamt-Elektronenanzahl
# und seine Standard-Visualisierung (Farbe/Größe im Browser).
ELEMENT_PROFIL = {
    "H":  {"geo": "KUGEL",     "valenzelektronen": 1, "radius": 0.18, "farbe": 0xffffff},
    "Li": {"geo": "KUGEL",     "valenzelektronen": 1, "radius": 0.25, "farbe": 0x9b59b6},
    "Be": {"geo": "HANTEL",    "valenzelektronen": 2, "radius": 0.28, "farbe": 0xd35400},
    "B":  {"geo": "DREIECK",   "valenzelektronen": 3, "radius": 0.28, "farbe": 0x1abc9c},
    "C":  {"geo": "TETRAEDER", "valenzelektronen": 4, "radius": 0.30, "farbe": 0x34495e},
    "N":  {"geo": "TETRAEDER", "valenzelektronen": 5, "radius": 0.30, "farbe": 0x2980b9},
    "O":  {"geo": "TETRAEDER", "valenzelektronen": 6, "radius": 0.30, "farbe": 0xe74c3c},
    "F":  {"geo": "TETRAEDER", "valenzelektronen": 7, "radius": 0.30, "farbe": 0x27ae60},
}

def generiere_wolken_vektoren(geo_typ, r=0.6):
    """Berechnet die rein geometrischen Richtungsvektoren für die Wolken."""
    if geo_typ == "KUGEL":
        return [np.array([0.0, 0.0, 0.0])]

    elif geo_typ == "HANTEL":
        return [np.array([0.0, 0.0, r]), np.array([0.0, 0.0, -r])]

    elif geo_typ == "DREIECK":
        return [
            np.array([0.0, r, 0.0]),
            np.array([r * np.sin(np.radians(120)), r * np.cos(np.radians(120)), 0.0]),
            np.array([-r * np.sin(np.radians(120)), r * np.cos(np.radians(120)), 0.0])
        ]

    elif geo_typ == "TETRAEDER":
        t_coords = [[1, 1, 1], [-1, -1, 1], [-1, 1, -1], [1, -1, -1]]
        return [np.array(c) / np.linalg.norm(c) * r for c in t_coords]

    return []


# ==============================================================================
# 2. DIE DATENSTRUKTUR FÜR ATOME UND DIE WELT
# ==============================================================================
class Kugelwolke:
    def __init__(self, id_num, elektronen, richtung_lokal):
        self.id = id_num
        self.elektronen = elektronen  # 1 = reaktiv (blau), 2 = frei (grau), 3 = gebunden (grün)
        self.richtung_lokal = richtung_lokal

class Atom:
    def __init__(self, uid, symbol, position=[0.0, 0.0, 0.0]):
        self.uid = uid
        self.symbol = symbol
        self.position = np.array(position, dtype=float)
        self.rotation = np.eye(3)
        self.wolken = []

        # Holen uns das Profil aus dem "Wörterbuch"
        profil = ELEMENT_PROFIL[symbol]
        vektoren = generiere_wolken_vektoren(profil["geo"])

        # Elektronen intelligent auf die Wolken verteilen
        ve_uebrig = profil["valenzelektronen"]
        anzahl_wolken = len(vektoren)

        # Jede Wolke bekommt erstmal mindestens 1 Elektron (Hundsche Regel vereinfacht)
        wolken_e = [0] * anzahl_wolken
        for i in range(ve_uebrig):
            wolken_e[i % anzahl_wolken] += 1

        for i, vec in enumerate(vektoren):
            self.wolken.append(Kugelwolke(id_num=i, elektronen=wolken_e[i], richtung_lokal=vec.tolist()))

    def to_dict(self):
        profil = ELEMENT_PROFIL[self.symbol]
        return {
            "uid": self.uid,
            "symbol": self.symbol,
            "position": self.position.tolist(),
            "rotation": self.rotation.flatten().tolist(),
            "radius": profil["radius"],
            "farbe": profil["farbe"],
            "wolken": [w.__dict__ for w in self.wolken]
        }

# Der Weltzustand startet jetzt komplett leer und dynamisch!
WELT_ZUSTAND = {}


# ==============================================================================
# 3. DIE API ENDPOINTS (Kurz, klar, funktional)
# ==============================================================================
@app.get("/status")
def get_status():
    return {uid: atom.to_dict() for uid, atom in WELT_ZUSTAND.items()}


@app.post("/erzeuge_atom")
def erzeuge_atom(daten: dict):
    """Fügt der Szene dynamisch ein neues Atom hinzu."""
    symbol = daten["symbol"]
    # Generiere eine eindeutige ID (z.B. C_a83f) und eine leicht zufällige Startposition
    uid = f"{symbol}_{uuid.uuid4().hex[:4]}"
    zufalls_pos = (np.random.rand(3) - 0.5) * 4.0

    WELT_ZUSTAND[uid] = Atom(uid, symbol, position=zufalls_pos.tolist())
    return {"status": "success", "uid": uid}

@app.post("/binde")
def binde_atome(daten: dict):
    atom_A = WELT_ZUSTAND[daten["atom_A_id"]]
    atom_B = WELT_ZUSTAND[daten["atom_B_id"]]
    wolken_A_ids = daten["wolken_A_ids"]
    wolken_B_ids = daten["wolken_B_ids"]

    anzahl_wolken = len(wolken_A_ids)

    # Lokale Vektoren extrahieren
    punkte_A_lokal = [np.array(atom_A.wolken[i].richtung_lokal) for i in wolken_A_ids]
    punkte_B_lokal = [np.array(atom_B.wolken[i].richtung_lokal) for i in wolken_B_ids]

    # Geometrische Sortierung (Verdrehungsschutz)
    if anzahl_wolken > 1:
        sortierte_B_lokal = []
        uebrige_B = list(punkte_B_lokal)
        for pA in punkte_A_lokal:
            # Zuordnung über den minimalen Abstand zur invertierten A-Wolke
            index_des_naechsten = min(range(len(uebrige_B)), key=lambda idx: np.linalg.norm((-pA) - uebrige_B[idx]))
            sortierte_B_lokal.append(uebrige_B.pop(index_des_naechsten))
        punkte_B_lokal = sortierte_B_lokal

    # 1. DYNAMISCHER ABSTAND NACH BINDUNGSORDNUNG
    if anzahl_wolken == 3:    fester_abstand = 1.02  # Dreifachbindung (Flächen-Teilung)
    elif anzahl_wolken == 2:  fester_abstand = 1.12  # Doppelbindung (Kanten-Teilung)
    else:                     fester_abstand = 1.25  # Einfachbindung (Punkt-Teilung)

    # Zentralatom-Schutz prüfen
    atom_A_schon_gebunden = any(w.elektronen == 3 for w in atom_A.wolken)

    # 2. BINDUNGSACHSE BESTIMMEN
    if atom_A_schon_gebunden:
        # Globale Richtung der ausgewählten Wolken von A bestimmt die Achse
        A_ausgerichtet = [atom_A.rotation @ p for p in punkte_A_lokal]
        zentrum_A_global = np.mean(A_ausgerichtet, axis=0)
        bindungs_richtung = zentrum_A_global / np.linalg.norm(zentrum_A_global)
    else:
        # Falls A frei ist, nutzen wir den aktuellen Einflugvektor von B
        aktuelle_differenz = atom_B.position - atom_A.position
        distanz = np.linalg.norm(aktuelle_differenz)
        bindungs_richtung = aktuelle_differenz / distanz if distanz > 1e-3 else np.array([1.0, 0.0, 0.0])

    # Position von Atom B fixieren
    atom_B.position = atom_A.position + bindungs_richtung * fester_abstand

    # 3. ROTATION FÜR ATOM A (falls ungebunden)
    if not atom_A_schon_gebunden:
        zentrum_A_lokal = np.mean(punkte_A_lokal, axis=0)
        if np.linalg.norm(zentrum_A_lokal) > 1e-5:
            z_norm = zentrum_A_lokal / np.linalg.norm(zentrum_A_lokal)
            v = np.cross(z_norm, bindungs_richtung)
            c = np.dot(z_norm, bindungs_richtung)
            if not np.allclose(v, 0):
                s = np.linalg.norm(v)
                kmat = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
                atom_A.rotation = np.eye(3) + kmat + kmat @ kmat * ((1 - c) / (s ** 2))

    # Globale Ausrichtung der bindenden A-Wolken berechnen
    A_global_wolken = [atom_A.rotation @ p for p in punkte_A_lokal]

    # 4. EXAKTE GEOMETRISCHE RESTRIKTIONEN (CONSTRAINTS) FÜR ROTATION B
    if anzahl_wolken == 1:
        # Einfachbindung: SVD für kollineare Ausrichtung der Partnerwolke
        B_ziel = -A_global_wolken[0]
        B_start = punkte_B_lokal[0]

        # Hilfs-SVD mit orthogonalen Vektoren zur Vermeidung von Singularitäten
        H = np.outer(B_start, B_ziel)
        U, S, Vt = np.linalg.svd(H)
        R_B = Vt.T @ U.T
        if np.linalg.det(R_B) < 0:
            Vt[-1, :] *= -1
            R_B = Vt.T @ U.T
        atom_B.rotation = R_B

    elif anzahl_wolken == 2:
        # Doppelbindung (Kanten-Teilung): Erzwinge Ebenen-Bedingung & Antiparisierung
        # Ziel: (W_B - M) == -(W_A - M)
        B_ziel = -np.array(A_global_wolken)
        B_start = np.array(punkte_B_lokal)

        # Orthogonaler Vektor (Normalenvektor der Bindungsebene) zur rigiden Fixierung der Rotation
        normalen_A = np.cross(A_global_wolken[0], A_global_wolken[1])
        normalen_B = np.cross(punkte_B_lokal[0], punkte_B_lokal[1])

        B_ziel = np.vstack([B_ziel, normalen_A])
        B_start = np.vstack([B_start, normalen_B])

        H = B_start.T @ B_ziel
        U, S, Vt = np.linalg.svd(H)
        R_B = Vt.T @ U.T
        if np.linalg.det(R_B) < 0:
            Vt[-1, :] *= -1
            R_B = Vt.T @ U.T
        atom_B.rotation = R_B

    elif anzahl_wolken == 3:
        # Dreifachbindung (Flächen-Teilung): Vollkommene Starrheit
        # Ziel: Die Wolken von B müssen exakt auf die invertierten Wolken von A zeigen
        B_ziel = -np.array(A_global_wolken)
        B_start = np.array(punkte_B_lokal)

        # 1. Basis-SVD-Rotation berechnen
        H = B_start.T @ B_ziel
        U, S, Vt = np.linalg.svd(H)
        R_B = Vt.T @ U.T
        if np.linalg.det(R_B) < 0:
            Vt[-1, :] *= -1
            R_B = Vt.T @ U.T

        # 2. DIREKTER ORIENTIERUNGS-CHECK DER BASIS-ROTATION
        # Wir transformieren das lokale Zentrum der 3 B-Wolken mit der reinen SVD-Matrix
        zentrum_B_nach_svd = np.mean([R_B @ p for p in punkte_B_lokal], axis=0)

        # Das Zentrum der B-Wolken MUSS in die entgegengesetzte Richtung der Bindungsachse
        # zeigen (also zurück zu Atom A). Wenn das Skalarprodukt positiv ist,
        # schaut das Atom "nach hinten" weg.
        if np.dot(zentrum_B_nach_svd, bindungs_richtung) > 0:
            # Das Atom hat sich weggedreht! Wir müssen es um 180 Grad um eine Achse drehen,
            # die ORTHOGONAL zur Bindungsachse steht, um es nach innen zu klappen.
            # Wir suchen uns einen beliebigen orthogonalen Vektor zur Bindungsachse:
            if abs(bindungs_richtung[0]) < 0.9:
                ortho_achse = np.array([1.0, 0.0, 0.0])
            else:
                ortho_achse = np.array([0.0, 1.0, 0.0])
            ortho_achse = np.cross(bindungs_richtung, ortho_achse)
            ortho_achse /= np.linalg.norm(ortho_achse)

            # Rodrigues-Rotation für 180 Grad um die orthogonale Achse
            K_ortho = np.array([
                [0, -ortho_achse[2], ortho_achse[1]],
                [ortho_achse[2], 0, -ortho_achse[0]],
                [-ortho_achse[1], ortho_achse[0], 0]
            ])
            # cos(180) = -1, sin(180) = 0
            R_flip_innen = np.eye(3) + 2 * (K_ortho @ K_ortho)
            R_B = R_flip_innen @ R_B

        # 3. JETZT ERST DIE TORSIONS-KORREKTUR UM DIE BINDUNGSACHSE (-60 Grad)
        # Da das Atom nun garantiert nach innen gerichtet ist, greift die Torsion fehlerfrei
        w = bindungs_richtung / np.linalg.norm(bindungs_richtung)
        alpha = np.radians(-60)

        cos_a = np.cos(alpha)
        sin_a = np.sin(alpha)
        K_achse = np.array([[0, -w[2], w[1]], [w[2], 0, -w[0]], [-w[1], w[0], 0]])
        R_torsion = np.eye(3) + sin_a * K_achse + (1 - cos_a) * (K_achse @ K_achse)

        # Finale Zuweisung
        atom_B.rotation = R_torsion @ R_B

@app.post("/reset")
def reset_simulation():
    WELT_ZUSTAND.clear()
    return {"status": "success"}

Server starten

In [26]:
import multiprocessing
import uvicorn
import nest_asyncio

# Erlaubt verschachtelte Event-Loops in Jupyter/Colab Notebooks
nest_asyncio.apply()

def start_api():
    # Wir übergeben direkt das 'app'-Objekt aus Zelle 1
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

# Server in einen separaten Hintergrundprozess auslagern
server_prozess = multiprocessing.Process(target=start_api)
server_prozess.start()
print("Backend-Server wurde erfolgreich im Hintergrund auf Port 8000 gestartet!")

Backend-Server wurde erfolgreich im Hintergrund auf Port 8000 gestartet!


Test server

In [18]:
#server_prozess.terminate()
#print("Server gestoppt.")

In [27]:
# 1. Installiere pyngrok
!pip install pyngrok

from pyngrok import ngrok

# ==========================================
# HIER FÜGST DU DEINEN AUTHTOKEN EIN
# ==========================================
# Ersetze 'DEIN_NGROK_AUTHTOKEN_HIER' mit dem langen Code von der ngrok-Webseite
NGROK_TOKEN = "1Y1jBALsPqgA8RQOJd4HDib3Ofh_6HgoK1dPV8iBcKbVvxbQb"

# Dem ngrok-System den Token mitteilen
ngrok.set_auth_token(NGROK_TOKEN)

# Kill any running ngrok processes from this session to free up tunnel slots
# ngrok.kill()

# 2. Jetzt erst den Tunnel zu Port 8000 öffnen
public_url = ngrok.connect(8000, "http")
print("Deine öffentliche API-Zustandsadresse für deine lokale UI lautet:")
print(f"{public_url.public_url}/status")

Deine öffentliche API-Zustandsadresse für deine lokale UI lautet:
https://7983-34-75-42-34.ngrok-free.app/status


In [28]:
from fastapi.responses import HTMLResponse

@app.get("/", response_class=HTMLResponse)
def read_root():
    # 1. Lies die lokale index.html ein
    with open("index.html", "r", encoding="utf-8") as f:
        html_content = f.read()

    # 2. Wir suchen NUR nach dem Variablen-Namen und überschreiben die Zuweisung komplett.
    # Egal ob in deiner index.html 'http...' oder 'https...' steht:
    # Wir zwingen das JavaScript dazu, die aktuelle Server-Adresse dynamisch zu ermitteln.

    # Wir ersetzen die alte Zeile komplett mit einer sauberen JavaScript-Anweisung:
    html_content = html_content.replace(
        "const API_URL = 'http://127.0.0.1:8000';",
        "const API_URL = window.location.origin;"
    )

    # Falls du in der index.html schon die https-ngrok-Adresse fest eingetragen hattest,
    # fangen wir das hier sicherheitshalber auch noch ab:
    if "const API_URL = 'https://" in html_content:
        # Wir suchen das Ende der Zeile und ersetzen sie sauber
        start_idx = html_content.find("const API_URL = 'https://")
        end_idx = html_content.find("';", start_idx) + 2
        alte_zeile = html_content[start_idx:end_idx]
        html_content = html_content.replace(alte_zeile, "const API_URL = window.location.origin;")

    return html_content

In [21]:
!pip install clifford
import pytest
import numpy as np
from clifford.g3c import * # Geometrische Algebra G3 für Multi-Vector-Wedge-Validierung

# Dummy-Strukturen für den Testlauf analog zur Server-Logik
def create_tetrahedron_vectors():
    t_coords = [[1, 1, 1], [-1, -1, 1], [-1, 1, -1], [1, -1, -1]]
    return [np.array(c) / np.linalg.norm(c) * 0.6 for c in t_coords]

def test_methane_volume_constraint():
    """Validiert den Tetraeder-Schwerpunkt und das gleichmäßige Volumen von Methan (CH4)."""
    # Kernzentrum C (0-Simplex)
    C = np.array([0.0, 0.0, 0.0])
    H_vektoren = create_tetrahedron_vectors()

    # Constraint 1: Schwerpunkt-Regel
    schwerpunkt = C - (1.0 / 4.0) * sum(H_vektoren)
    assert np.allclose(schwerpunkt, 0.0, atol=1e-5)

    # GA Wedge-Produkt Validierung für identische Sub-Tetraeder-Volumina
    # Übertragung der Vektoren in die Algebra
    h1, h2, h3, h4 = [eo + h[0]*e1 + h[1]*e2 + h[2]*e3 + 0.5*np.dot(h,h)*einf for h in H_vektoren]
    c_ga = eo # Ursprung

    vol1 = (h1 - c_ga) ^ (h2 - c_ga) ^ (h3 - c_ga)
    vol2 = (h2 - c_ga) ^ (h3 - c_ga) ^ (h4 - c_ga)

    assert np.allclose(abs(vol1), abs(vol2), atol=1e-4)

def test_ethene_planar_constraint():
    """Überprüft die Kantenparallelität und Planarität bei einer Doppelbindung (Ethen)."""
    # Vektoren zweier gebundener Kohlenstoffe (Kanten-Teilung)
    # Wenn die SVD-Engine korrekt rechnet, müssen die Richtungsvektoren koplanar sein
    W_A1 = np.array([0.3, 0.4, 0.5])
    W_A2 = np.array([0.3, -0.4, 0.5])

    # Perfekt gespiegeltes System durch Kanten-Teilung-Constraint
    W_B1 = -W_A1
    W_B2 = -W_A2

    # Det-Bedingung für Koplanarität (Klassische Vektorrechnung)
    matrix = np.vstack([W_A1, W_A2, W_B1])
    assert np.allclose(np.linalg.det(matrix), 0.0, atol=1e-5)

In [22]:
create_tetrahedron_vectors()

[array([0.34641016, 0.34641016, 0.34641016]),
 array([-0.34641016, -0.34641016,  0.34641016]),
 array([-0.34641016,  0.34641016, -0.34641016]),
 array([ 0.34641016, -0.34641016, -0.34641016])]

In [23]:
test_methane_volume_constraint()

In [24]:
test_ethene_planar_constraint()